# 06 · 스트림과 비동기 처리

> **CuPy 2일 집중 코스 — Day 1 / 단원 4 (스트림과 비동기 처리) — Day 1 마무리**

단원 3(메모리)과 한 묶음으로, "데이터 이동을 어떻게 **숨기나(overlap)**"가 주제입니다.
스트림·이벤트로 연산과 전송을 겹치고, 이중 버퍼 파이프라인·CUDA Graph까지 다룹니다.

## 학습 목표
- 스트림(순서 실행)과 다른 스트림 간 **오버랩**, 이벤트 동기화를 이해한다.
- **이중 버퍼 청크 파이프라인**으로 전송-연산을 겹치고 속도를 측정한다.
- 이벤트로 구간을 세분 측정하고, 스트림 수를 스케일링한다.
- (심화) **CUDA Graph 캡처**로 런치 오버헤드를 줄이고, 멀티-GPU를 맛본다.

## 목차
1. [스트림이란](#1)
2. [이벤트로 구간 측정](#2)
3. [다중 스트림 + 이벤트 동기화](#3)
4. [비동기 전송 & pinned memory](#4)
5. [이중 버퍼 청크 오버랩 파이프라인](#5)
6. [이벤트 세분 타이밍](#6)
7. [스트림 수 스케일링](#7)
8. [NVTX/프로파일 연계 + Power Iteration](#8)
9. [(심화) CUDA Graph 캡처](#9)
10. [(심화) 멀티-GPU](#10)
11. [체크포인트](#11)

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from course_utils import print_env, bench, gpu_ms, cpu_ms, print_bench, compare
print_env()

<a id="1"></a>
## 1. 스트림이란

**스트림(stream)** 은 *순서대로 실행되는 디바이스 작업의 사슬*입니다. 같은 스트림은 순서 보장,
**다른 스트림끼리는 겹쳐(overlap)** 실행될 수 있습니다. 지정 안 하면 기본 스트림을 씁니다(`cp.cuda.get_current_stream()`).

<img src="images/figures/new_stream_concept.png" width="560">



In [ ]:
A = cp.cuda.Stream(non_blocking=True)
a = cp.random.random(10_000_000, dtype=cp.float32)
with A:
    s = (cp.sin(a)+1).sum()
A.synchronize()
print('A 스트림 결과:', float(s))

<a id="2"></a>
## 2. 이벤트로 구간 측정

이벤트는 스트림 위 시점 표식입니다. 두 이벤트 사이 시간으로 GPU 구간을 정확히 잽니다.

In [ ]:
a = cp.random.random(30_000_000, dtype=cp.float32)
st=cp.cuda.Event(); ed=cp.cuda.Event()
st.record(); y=(cp.sin(a)+1).sum(); ed.record(); ed.synchronize()
print('구간:', cp.cuda.get_elapsed_time(st,ed),'ms')

<a id="3"></a>
## 3. 다중 스트림 + 이벤트 동기화

여러 스트림에 독립 작업을 올리면 겹칠 수 있고, 스트림 간 의존성은 **이벤트**(`record`/`wait_event`)로 표현합니다.

<img src="images/figures/new_streams_events.png" width="620">



In [ ]:
a1=cp.random.random(20_000_000,dtype=cp.float32); a2=cp.random.random(20_000_000,dtype=cp.float32)
s1=cp.cuda.Stream(non_blocking=True); s2=cp.cuda.Stream(non_blocking=True)
e=cp.cuda.Event()
with s1:
    t=cp.sin(a1); e.record(s1)
with s2:
    s2.wait_event(e)          # s2는 e 이후 진행
    out=(t+cp.cos(a2)).sum()
s2.synchronize()
print('의존성 결합:', float(out))

<a id="4"></a>
## 4. 비동기 전송 & pinned memory

전송은 기본 블로킹입니다. CuPy 13+는 `cp.asarray(x, blocking=False)`/`cp.asnumpy(x, blocking=False)`로 비동기 전송을 합니다.
겹침 효과를 보려면 host 버퍼가 **pinned(page-locked)** 여야 합니다(`cp.cuda.alloc_pinned_memory`).

- Pinned Memory (Page-Locked Memory) 개념 정리: 
"OS가 마음대로 메모리 주소를 바꾸거나 디스크로 쫓아내지 못하도록, RAM의 특정 위치에 '고정해 둔' 메모리 영역"
- 왜 Pinned Memory가 필요할까? (배경 원리)
우리가 평소에 쓰는 일반 Host 메모리(Pageable Memory)는 OS가 효율적인 관리(가상 메모리)를 위해 필요시 디스크(Swap)에 내보내거나 실제 물리 주소를 바꿉니다.
- 일반 메모리 (Pageable Memory)를 GPU로 보낼 때 일어나는 일: 
CPU가 일반 RAM에 있는 데이터를 Pinned Memory(임시 버퍼)로 복사함 (추가 복사 단계 발생!)
Pinned Memory에 올라간 데이터를 DMA(Direct Memory Access)를 통해 GPU로 전송함
- Pinned Memory를 직접 만들어 보낼 때 일어나는 일: 
처음부터 Pinned Memory에 데이터를 생성함. 
복사 단계 없이 곧바로 DMA를 통해 GPU로 직통 전송!
- 비동기 전송(Asynchronous Transfer)과의 관계:
비동기 전송(Stream 전송)을 가능하게 만드는 필수 전제 조건이 바로 Pinned Memory입니다. 
- 일반 메모리 전송 (동기식): 
CPU는 GPU로 데이터 복사(Stage 과정)가 끝날 때까지 아무 일도 못 하고 대기(Blocking)
- Pinned 메모리 전송 (비동기식): 
GPU 전송을 DMA(하드웨어 엔진)에 맡겨버리고, CPU는 기다리지 않고 다음 파이썬 코드를 즉시 실행합니다. 
[데이터 전송]과 [GPU 연산/CPU 연산]을 동시에 진행(Overlap)할 수 있게 됩니다.
- pinned memory 사용시 주의점: 
실제 물리 RAM 용량을 그대로 점유해 버리기 때문에, OS나 다른 프로세스가 사용할 수 있는 실질 메모리 양이 대폭 줄어듭니다. 
Pinned Memory를 너무 과도하게 할당하면, 정작 OS 핵심 프로세스나 파이썬 메인 프로그램이 쓸 메모리가 부족해집니다. 
전체 Host RAM의 50%~70% 이상을 Pinned Memory로 채우지 않도록 주의하세요.


In [ ]:
n=8_000_000; itemsize=np.dtype(np.float32).itemsize
pinned=cp.cuda.alloc_pinned_memory(n*itemsize)
h=np.frombuffer(pinned,dtype=np.float32,count=n); h[:]=np.random.rand(n).astype(np.float32)
s=cp.cuda.Stream(non_blocking=True)
with s:
    try: d=cp.asarray(h, blocking=False)
        # 1. 최신 CuPy인 경우: 비동기(Non-blocking)로 GPU 메모리(pinned)에 바로 복사 시도
    except TypeError: d=cp.asarray(h)
        # 2. 구버전 CuPy인 경우: 'blocking' 인자가 없으므로 TypeError 발생 -> 동기(Blocking) 전송으로 처리
    y=(d*2+1).sum()
s.synchronize(); print('pinned 비동기 전송 결과:', float(y))

<a id="5"></a>
## 5. 이중 버퍼 청크 오버랩 파이프라인
스트림의 **핵심 활용**입니다. 큰 데이터를 청크로 나눠 여러 스트림에 분산하면,
한 청크의 전송과 다른 청크의 연산이 **겹쳐** 전체 시간이 줄어듭니다. 먼저 순차 기준선과 비교합니다.

- 순차 실행 방식 (sequential)
```
* 특징: H->D 전송 중에는 GPU 코어가 놀고(Idle), GPU 연산 중에는 PCIe 전송 통로(DMA Engine)가 놉니다.
[시간 흐름 (Time) ----------------------------------------------------------------------------------------►]

청크 0 : | H->D 전송 | === GPU 연산 === | D->H 회수 |
청크 1 :                                           | H->D 전송 | === GPU 연산 === | D->H 회수 |
청크 2 :                                                                                      | H->D ...
------------------------------------------------------------------------------------------------------------
``


- 비동기 스트림 방식 (chunked(nstreams=3))
 `` 
* Pinned Memory + 다중 Stream 활용
* 한 청크가 GPU 연산을 하는 동안, 다른 청크는 PCIe 통로를 통해 전송(Overlap)을 동시 진행합니다.
[시간 흐름 (Time) ----------------------------------------------------------------------------------------►]

Stream 0 (청크 0) : | H->D (0) | ====== GPU 연산 (0) ====== | D->H (0) |
Stream 1 (청크 1) :            | H->D (1) | ====== GPU 연산 (1) ====== | D->H (1) |
Stream 2 (청크 2) :                       | H->D (2) | ====== GPU 연산 (2) ====== | D->H (2) |
Stream 0 (청크 3) :                                  | H->D (3) | ====== GPU 연산 (3) ====== | D->H (3) |
------------------------------------------------------------------------------------------------------------
```



In [ ]:
N = 64_000_000      # 전체 데이터의 개수 (6,400만 개)
CH = 4_000_000      # 한 번에 처리할 청크(Chunk)의 크기 (400만 개)
itemsize = np.dtype(np.float32).itemsize  # float32의 바이트 크기 (4바이트)
pin=cp.cuda.alloc_pinned_memory(N*itemsize)
# 고정 메모리(Pinned Memory) 할당: GPU 비동기 전송의 핵심입니다. OS가 페이지 아웃(Page-out)시키지 못하도록 메모리 상에 딱 고정시켜 둔 특수 메모리 영역을 CuPy를 통해 할당합니다. 이 영역을 쓰면 CPU와 GPU 간의 데이터 전송 속도가 훨씬 빨라집니다.
H=np.frombuffer(pin,dtype=np.float32,count=N); H[:]=np.random.rand(N).astype(np.float32)
# 호스트 배열 연결: 방금 할당한 고정 메모리(pin)를 껍데기로 삼아, NumPy 배열 H를 생성합니다. 그리고 이 배열에 임의의 랜덤 값(0~1 사이)을 채워 넣습니다. 이제 H는 고정 메모리에 매핑된 상태입니다.
def work(d): return cp.sqrt(d*d+1.0)

# 기준선(순차): 한 청크씩 전송->연산->회수 (겹침 없음)
def sequential():
    out=np.empty(N,np.float32)
# s0는 0부터 시작해 CH(4,000,000)씩 커지며 N 직전까지 반복하는 루프입니다.
    for s0 in range(0,N,CH):
        d=cp.asarray(H[s0:s0+CH]);
# 호스트 메모리의 s0부터 s0+CH 범위의 조각(16MB)을 GPU 메모리로 복사(전송)합니다. 이 작업이 끝날 때까지 CPU는 다음 줄로 넘어가지 않고 기다립니다(동기식).
        r=work(d); 
# GPU로 전송된 데이터 d를 가지고 위에서 정의한 work() 연산을 수행합니다. 연산 결과인 r 역시 GPU 메모리에 머무릅니다.
        out[s0:s0+CH]=cp.asnumpy(r)
# GPU 메모리에 있는 결과 r을 다시 호스트(CPU)의 out 배열 슬라이스 영역으로 복사하여 가져옵니다.
    return out
print_bench(bench(sequential, n_repeat=5, name='sequential'))


**연습 — 이중 버퍼 오버랩 직접 구현**: `H`를 `CH` 청크로 나눠 `nstreams`개 스트림에 분산하고,
각 청크에서 `asarray(blocking=False)`→`work`→`asnumpy(blocking=False)`로 처리해 **순차 대비 속도**를 비교하세요.

In [ ]:
def chunked(nstreams=3):
    # TODO: nstreams개 Stream(non_blocking=True) 생성
    #       각 청크를 streams[i%nstreams]에서 asarray(blocking=False)->work->asnumpy(blocking=False)
    #       모든 스트림 synchronize 후 out 반환
    raise NotImplementedError

# print_bench(bench(lambda: chunked(3), n_repeat=5, name='chunked(3)'))
# print('speedup:', round(cpu_ms(bench(sequential))/cpu_ms(bench(lambda: chunked(3))),2))

<details><summary>💡 해답 보기</summary>

```python
def chunked(nstreams=3):
    out=np.empty(N,np.float32)
    streams=[cp.cuda.Stream(non_blocking=True) for _ in range(nstreams)]
    for i,s0 in enumerate(range(0,N,CH)):
        with streams[i%nstreams]:
            try: d=cp.asarray(H[s0:s0+CH], blocking=False)
            except TypeError: d=cp.asarray(H[s0:s0+CH])
            r=work(d)
            try: out[s0:s0+CH]=cp.asnumpy(r, blocking=False)
            except TypeError: out[s0:s0+CH]=cp.asnumpy(r)
    for st in streams: st.synchronize()
    return out

for k in [1,2,4]:
    print_bench(bench(lambda k=k: chunked(k), n_repeat=5, name=f'chunked({k})'))
# 보통 2~3 스트림에서 이득이 포화됩니다.
```
</details>

<a id="6"></a>
## 6. 이벤트 세분 타이밍

전송 구간과 연산 구간을 이벤트로 분리 측정하면 어디가 병목인지 보입니다.

In [ ]:
e0=cp.cuda.Event(); e1=cp.cuda.Event(); e2=cp.cuda.Event()
e0.record()
d=cp.asarray(H[:CH])      # 전송
e1.record()
r=work(d).sum()           # 연산
e2.record(); e2.synchronize()
print(f'transfer {cp.cuda.get_elapsed_time(e0,e1):.3f} ms | compute {cp.cuda.get_elapsed_time(e1,e2):.3f} ms')

<a id="7"></a>
## 7. 스트림 수 스케일링

독립 연산을 1·2·4·8 스트림에 나눠 올리고 시간을 비교합니다. 단일 커널이 GPU를 이미 채우면 이득이 작습니다.

In [ ]:
arrs=[cp.random.random(20_000_000,dtype=cp.float32) for _ in range(8)]
def run_streams(k):
    streams=[cp.cuda.Stream(non_blocking=True) for _ in range(k)]; outs=[]
    for i,a in enumerate(arrs):
        with streams[i%k]: outs.append((cp.sin(a)+1).sum())
    for st in streams: st.synchronize()
    return outs
for k in [1,2,4,8]:
    print_bench(bench(lambda k=k: run_streams(k), n_repeat=10, name=f'{k} stream(s)'))

<a id="8"></a>
## 8. NVTX/프로파일 연계 + Power Iteration

`cupyx.profiler.time_range`로 구간을 NVTX 라벨링하면 Nsight Systems 타임라인에서 오버랩·idle을 볼 수 있습니다(단원 3 도구).
단원 3의 Power Iteration으로 **동기화 빈도**의 영향을 다시 확인합니다(잦은 `float()`=잦은 host 동기화).
`!nsys profile python ./cupy/06_8.py` 실행후, 출력파일(확장자 nsys-rep) 다운로드 하신뒤, 로컬 컴퓨터에서 nsight 프로그램을 실행해서 타임라인을 확인해보세요. `time_range`로 구간을 라벨링하세요.


In [ ]:
%%writefile 06_8.py
import cupy as cp
from cupyx.profiler import time_range, benchmark
from cupy.cuda import profiler

# 1. 테스트용 행렬 준비
M = cp.random.random((1500, 1500), dtype=cp.float32)
A = (M + M.T) / 2

def power_iter_sync(A, iters=300, check_every=1):
    xp = cp.get_array_module(A)
    x = xp.ones(A.shape[0], dtype=A.dtype)
    for i in range(iters):
        y = A @ x
        nrm = xp.linalg.norm(y)
        x = y / nrm
        if i % check_every == 0:
            _ = float(nrm)   # host 동기화 (CPU-GPU 병목 유발)
    return x

# 프로파일링 명시적 시작
profiler.start()

# [구간 1] 매 스텝 동기화
with time_range('pi_check1', color_id=0):
    # bench 실행
    res1 = benchmark(lambda: power_iter_sync(A, check_every=1), n_repeat=5, name='sync 매 스텝')
    # ★ 핵심: NVTX 구간(time_range)이 닫히기 전에 GPU 연산이 모두 끝날 때까지 대기
    cp.cuda.Device().synchronize()

print(res1)

# [구간 2] 50스텝마다 동기화
with time_range('pi_check50', color_id=1):
    res2 = benchmark(lambda: power_iter_sync(A, check_every=50), n_repeat=5, name='sync 50스텝마다')
    # ★ 핵심: NVTX 구간이 닫히기 전 GPU 동기화
    cp.cuda.Device().synchronize()

print(res2)

# 프로파일링 종료 및 버퍼 Flush
profiler.stop()

<details><summary>(선택) Nsight Systems 프로파일링 워크플로</summary>

```python
# %%writefile pi.py  로 스크립트 저장 후 터미널에서:
# nsys profile --capture-range=cudaProfilerApi --capture-range-end=stop -o pi python pi.py
# 생성된 pi.nsys-rep 를 Nsight Systems GUI / Perfetto 에서 열어 타임라인 확인
```
`with cupyx.profiler.profile():` 블록 안에서만 캡처되며, `time_range` 라벨이 타임라인에 표시됩니다.
</details>

<a id="9"></a>
## 9. (심화) CUDA Graph 캡처

📖 [`cupy.cuda.Graph`](https://docs.cupy.dev/en/stable/reference/generated/cupy.cuda.Graph.html) — 반복되는 스트림 작업 시퀀스를 **그래프로 캡처**해 한 번에 실행하면 **커널 런치 오버헤드**가 줄어듭니다.
`stream.begin_capture()` … `g = stream.end_capture()` … `g.launch()`. 캡처 중에는 **host 동기 전송 금지**입니다.

- 일반적으로 CuPy(또는 CUDA) 코드를 실행할 때, CPU는 GPU에게 "A 커널(함수)을 실행해", 그다음 "B 커널을 실행해"라고 매번 명령을 내립니다. 이 명령을 내리는 시간(CPU-GPU 통신 시간)을 커널 런치 오버헤드(Kernel Launch Overhead)라고 합니다.
- 작업 단위가 크다면 이 오버헤드가 무시할 만하지만, 연산이 매우 짧고 자잘한 커널을 수천~수만 번 반복해서 호출해야 한다면(예: 딥러닝 학습 루프, 반복적인 물리 시뮬레이션) 연산 시간보다 명령을 내리는 대기 시간이 더 길어지는 병목 현상이 발생합니다.
- CUDA Graph는 이 문제를 해결합니다. 일련의 GPU 작업(메모리 복사, 연산 등)을 실행하는 대신 "기록(Capture)"하여 하나의 거대한 작업 흐름도(Graph)로 묶어둡니다. 이후에는 CPU가 "저장된 그래프 실행해"라고 단 한 번만 명령하면 되므로 런치 오버헤드가 획기적으로 줄어듭니다.
- 그래프를 캡처하는 구간(begin_capture ~ end_capture)은 연산을 실행하는 것이 아니라 레시피를 적는 시간입니다. 따라서 GPU가 연산을 끝낼 때까지 기다렸다가 결과값을 CPU로 가져오는(Host Synchronization) 코드가 캡처 구간 안에 있으면 에러가 발생하거나 캡처가 중단됩니다.
   * 캡처 중 절대 하면 안 되는 행동: 
     * .get() 또는 cp.asnumpy() 호출: GPU 배열을 CPU 넘파이 배열로 변환하는 행위는 동기화를 강제합니다.
     * 조건문(if)에 GPU 값 사용: if a.sum() > 0: 와 같은 코드를 쓰면, 조건을 평가하기 위해 CPU가 GPU의 sum() 결과를 기다려야 하므로 동기화가 발생합니다.
     * stream.synchronize() 호출: 명시적인 동기화 명령이므로 당연히 금지됩니다.
- 모든 코드에 그래프를 적용할 필요는 없습니다. 다음과 같은 상황에서 성능 향상을 보입니다.
   * 반복 횟수가 매우 많은 루프: AI 모델의 에폭(Epoch) 반복, MCMC 샘플링 등 동일한 구조의 연산이 수백 번 이상 반복될 때.
   * 커널 크기가 작을 때: 개별 행렬 곱셈이나 덧셈 연산 자체는 1ms 이내로 끝나는데, 이를 연속해서 호출해야 할 때.
   * 제어 흐름(Control Flow)이 고정되어 있을 때: if-else 조건에 따라 연산 그래프 구조 자체가 매번 바뀌는 동적 모델(Dynamic Graph)에는 적용하기 어렵습니다.

In [ ]:
# 작은 커널을 여러 번 실행하는 반복 시퀀스 (런치 오버헤드 지배)
x=cp.random.random(1_000_000,dtype=cp.float32)
def many_small():
    for _ in range(50): cp.add(x,1.0,out=x)

s=cp.cuda.Stream(non_blocking=True)
try:
    with s:
        s.begin_capture()
        for _ in range(50): cp.add(x,1.0,out=x)
        g=s.end_capture()
    print_bench(bench(many_small, n_repeat=50, name='반복 런치'))
    print_bench(bench(lambda: g.launch(), n_repeat=50, name='CUDA Graph'))
except (AttributeError, RuntimeError) as ex:
    print('이 환경에서는 그래프 캡처를 건너뜁니다:', ex)

<a id="10"></a>
## 10. (심화) 멀티-GPU

GPU가 여러 개면 `with cp.cuda.Device(i):` 로 장치를 전환합니다. 연산 입력은 같은 장치에 있어야 합니다.

In [ ]:
ngpu=cp.cuda.runtime.getDeviceCount()
print('GPU 개수:', ngpu)
if ngpu>=2:
    with cp.cuda.Device(0): a0=cp.random.random(1_000_000,dtype=cp.float32); s0=float(a0.sum())
    with cp.cuda.Device(1): a1=cp.random.random(1_000_000,dtype=cp.float32); s1=float(a1.sum())
    print('device0 sum',s0,'| device1 sum',s1)
else:
    print('단일 GPU 환경 — 멀티-GPU 예제는 건너뜁니다.')

<a id="11"></a>
## 11. 체크포인트

- [ ] 스트림(순서)과 다른 스트림 간 오버랩, 이벤트 동기화를 이해했다
- [ ] **이중 버퍼 청크 파이프라인**으로 전송-연산을 겹쳐 속도를 높였다
- [ ] 이벤트로 전송 vs 연산 구간을 분리 측정했다
- [ ] 스트림 수 스케일링과 동기화 빈도의 영향을 확인했다
- [ ] (심화) CUDA Graph 캡처/멀티-GPU를 시도했다

**Day 1 완료!** 다음은 **`day1_capstone`**(통합 실습) → Day 2 **`07_cupy_kernels`**.